In [0]:
-- MAGIC ## Step 1: Create Gold Master Table

-- COMMAND ----------

CREATE OR REPLACE TABLE enterprise.gold_ingestion.gold_market_macro_daily
USING DELTA
PARTITIONED BY (trade_date)
COMMENT 'Analytics-ready daily table fusing crypto OHLCV and macroeconomic indicators with forward-filled point-in-time correctness.'
AS
WITH mkt AS (
  -- 1. Crypto Market Data (7x24 Continuous Calendar)
  -- Downsampling 1m OHLC to Daily OHLCV
  SELECT
    source,
    symbol,
    date(bar_start_ts) AS trade_date,
    first_value(open) OVER (PARTITION BY source, symbol, date(bar_start_ts) ORDER BY bar_start_ts ASC) AS daily_open,
    max(high) AS daily_high,
    min(low) AS daily_low,
    last_value(close) OVER (PARTITION BY source, symbol, date(bar_start_ts) ORDER BY bar_start_ts ASC) AS daily_close,
    sum(volume) AS daily_volume
  FROM enterprise.silver_ingestion.slv_crypto_ohlc_1m
  GROUP BY 
    source, symbol, date(bar_start_ts), open, close, bar_start_ts 
    -- Note: window functions in GROUP BY in Spark require careful handling, 
    -- Alternative cleaner OHLC aggregation strategy:
),
mkt_agg AS (
    SELECT 
        source,
        symbol,
        date(bar_start_ts) AS trade_date,
        min_by(open, bar_start_ts) AS daily_open,
        max(high) AS daily_high,
        min(low) AS daily_low,
        max_by(close, bar_start_ts) AS daily_close,
        sum(volume) AS daily_volume
    FROM enterprise.silver_ingestion.slv_crypto_ohlc_1m
    GROUP BY source, symbol, date(bar_start_ts)
),
fx AS (
  -- 2. ECB FX Data (Workdays Only)
  SELECT
    rate_date,
    fx_rate AS eurusd_rate
  FROM enterprise.silver_ingestion.slv_macro_ecb_fx_daily
  WHERE base_ccy = 'EUR' AND quote_ccy = 'USD'
),
fed_dff AS (
  -- 3. FRED Fed Funds Rate (Workdays Only)
  SELECT
    obs_date,
    value AS fedfunds_rate
  FROM enterprise.silver_ingestion.slv_macro_fred_series
  WHERE series_id = 'DFF'
),
fed_cpi AS (
  -- 4. FRED CPI (Monthly updates, usually posted on the 1st)
  SELECT
    obs_date,
    value AS cpi_value
  FROM enterprise.silver_ingestion.slv_macro_fred_series
  WHERE series_id = 'CPIAUCSL'
),
fed_dxy AS (
  -- 5. FRED Dollar Index (Workdays Only)
  SELECT
    obs_date,
    value AS dxy_value
  FROM enterprise.silver_ingestion.slv_macro_fred_series
  WHERE series_id = 'DTWEXBGS'
),
joined_raw AS (
  -- 6. Left Join onto the continuous Crypto Calendar
  -- Weekends and non-month-start dates will yield NULLs for macro metrics here.
  SELECT
    m.source,
    m.symbol,
    m.trade_date,
    m.daily_open,
    m.daily_high,
    m.daily_low,
    m.daily_close,
    m.daily_volume,
    fx.eurusd_rate as raw_fx,
    fed_dff.fedfunds_rate as raw_dff,
    fed_cpi.cpi_value as raw_cpi,
    fed_dxy.dxy_value as raw_dxy
  FROM mkt_agg m
  LEFT JOIN fx      ON fx.rate_date = m.trade_date
  LEFT JOIN fed_dff ON fed_dff.obs_date = m.trade_date
  LEFT JOIN fed_cpi ON fed_cpi.obs_date = m.trade_date
  LEFT JOIN fed_dxy ON fed_dxy.obs_date = m.trade_date
)

-- 7. Final Projection: Impute NULLs using Point-In-Time Forward Fill 
-- 'TRUE' within LAST_VALUE tells Spark to IGNORE NULLS.
SELECT
  source,
  symbol,
  trade_date,
  daily_open,
  daily_high,
  daily_low,
  daily_close,
  daily_volume,
  
  -- Fill missing weekend FX rates with Friday's rate
  LAST_VALUE(raw_fx, TRUE) OVER (
    PARTITION BY source, symbol 
    ORDER BY trade_date 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS eurusd_rate,
  
  -- Fill missing weekend/holiday fed funds rates
  LAST_VALUE(raw_dff, TRUE) OVER (
    PARTITION BY source, symbol 
    ORDER BY trade_date 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS fedfunds_rate,

  -- Carry forward the monthly CPI values for all 30/31 days in the month until a new one is observed
  LAST_VALUE(raw_cpi, TRUE) OVER (
    PARTITION BY source, symbol 
    ORDER BY trade_date 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS cpi_value,

  LAST_VALUE(raw_dxy, TRUE) OVER (
    PARTITION BY source, symbol 
    ORDER BY trade_date 
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ) AS dxy_value,
  
  current_timestamp() AS mart_built_ts
FROM joined_raw;

-- COMMAND ----------

-- MAGIC %md
-- MAGIC ## Step 2: Verification

-- COMMAND ----------

-- View the fused dataset ordered by most recent day
SELECT * 
FROM enterprise.gold_ingestion.gld_market_macro_daily
WHERE symbol = 'BTC-USD'
ORDER BY trade_date DESC
LIMIT 50;